# DYNAMOS E3 on FABRIC — slice, cluster, and DYNAMOS install

Adapted from Scattered-Directive `fabric/kubernetes.ipynb` (working run
2025-11-20: all six nodes on CIEN host `cien-w1`). This copy targeted CIEN for the smoke; the counted-run site is MAX
(max-w4) since 2026-09-26 — see the workbench run-design SPLIT & SITE
addendum. It targets the
DYNAMOS-tobias fork; node scripts live in `node_scripts/` next to this
notebook.

Phases:
1. **Pre-flight** — go/no-go checks before anything is created
2. **Slice** — 6 nodes on MAX (max-w4), FABNetv4, 2-week lease
3. **NVMe** — format + mount the NVMe drive(s)
4. **Node prep** — base packages on every node
5. **Kubespray** — Kubernetes cluster from the control node
6. **Provisioner** — local-path-provisioner (default StorageClass + gateway-state backing)
7. **DYNAMOS** — repo delivery, etcd seed, install
8. **Teardown** — uninstall / cleanup / slice delete

## FABlib API references

- [fablib.list_sites](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.list_sites)
- [fablib.new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice)
- [slice.add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node)
- [slice.submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit)
- [slice.get_nodes](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.get_nodes)
- [node.execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute)
- [slice.delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete)

In [ ]:
import datetime
import json

# The SD pipeline this notebook is adapted from was written against
# fablib 1.9.x; the FABRIC JupyterHub now serves 2.0.8. Cells below add
# explicit None-checks where the 2.x singular getters return None instead
# of raising.
import fabrictestbed_extensions
print(f"fabrictestbed_extensions version: {fabrictestbed_extensions.__version__}")

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()

fablib.show_config();

## Phase 1 — Pre-flight (go/no-go)

Run this **before** creating the slice. It stops if the pinned image is not
offered, and prints site + worker-host capacity so you can confirm `max-w4`
has free NVMe (the `server` node requests an `NVME_P4510`). If `max-w4` is
full, pick another qualifying MAX worker (max-w5 was equal at the 26.09
sweep) in the node config or fall back to GATECH/FIU (keep the NVMe
component either way) — do not submit blind.

In [ ]:
%%time
image = "default_ubuntu_24"

image_names = fablib.get_image_names()
if image not in image_names:
    print("Available images:")
    print(image_names)
    raise RuntimeError(f"image {image!r} is not offered by this FABRIC deployment — stop here")
print(f"OK: image {image!r} is available")

# Site capacity, including NVMe.
fablib.list_sites(fields=['name', 'state', 'cores_available', 'ram_available',
                          'disk_available', 'nvme_available', 'nvme_capacity'])
fablib.show_site('MAX')

# Worker hosts: confirm max-w4 has free NVMe before submitting.
try:
    hosts = fablib.list_hosts(output='list', quiet=True)
    for host in hosts:
        if 'max-' in str(host.get('name', '')).lower():
            print(host)
except Exception as e:
    print(f"list_hosts failed ({e}); check host capacity on the FABRIC portal instead")

## Phase 2 — Slice

Six nodes, all pinned to MAX host `max-w4` (site picked 2026-09-26; the
single-host pinning itself matches the SD working run).
One FABNetv4 L3 network per site carries the cluster traffic.

Submit asks for the 2-week maximum lease explicitly — a bare submit gets the
default 24h lease and the slice dies the next day.

**Patience:** polling for ~6–7 minutes (~14 retries) during submit is normal.
If it looks hung, verify the nodes respond via `node.execute(...)` before
deleting and recreating anything.

In [ ]:
%%time
# New name: the CIEN smoke slice keeps 'DYNAMOS-on-FABRIC' until teardown.
slice_name = 'DYNAMOS-E3-MAX'
image = "default_ubuntu_24"

# Please adhere to the following regex for naming: /[a-z][a-z0-9]+/
#
# "nvme": OPEN SCOPE DECISION — server-only NVMe is the default here; the
# alternative is per-steward NVMe (also set nvme=True on clientone/two/three).
# Keep in sync with the gateway-state nodePathMap in
# node_scripts/install_localpath_provisioner.sh.

node_configurations = [
    {
        "type": "control",
        "cores": 2,
        "ram": 8,
        "disk": 20,
        "site": "MAX",
        "host": "max-w4.fabric-testbed.net",
        "nvme": False,
    },
    {
        "type": "dynamos",
        "cores": 4,
        "ram": 16,
        "disk": 20,
        "site": "MAX",
        "host": "max-w4.fabric-testbed.net",
        "nvme": False,
    },
    {
        "type": "agent",
        "name": "server",
        "cores": 4,
        "ram": 16,
        "disk": 20,
        "site": "MAX",
        "host": "max-w4.fabric-testbed.net",
        "nvme": True,
    },
    {
        "type": "agent",
        "name": "clientone",
        "cores": 4,
        "ram": 16,
        "disk": 20,
        "site": "MAX",
        "host": "max-w4.fabric-testbed.net",
        "nvme": False,
    },
    {
        "type": "agent",
        "name": "clienttwo",
        "cores": 8,
        "ram": 16,
        "disk": 20,
        "site": "MAX",
        "host": "max-w4.fabric-testbed.net",
        "nvme": False,
    },
    {
        "type": "agent",
        "name": "clientthree",
        "cores": 4,
        "ram": 16,
        "disk": 20,
        "site": "MAX",
        "host": "max-w4.fabric-testbed.net",
        "nvme": False,
    }
]

sites = list(set([configuration["site"] for configuration in node_configurations]))
agents = [configuration["name"] for configuration in node_configurations if configuration["type"] == "agent"]
thirdparties = [configuration["name"] for configuration in node_configurations if configuration["type"] == "thirdparty"]

def create_node(slice, configuration):
    if (configuration["type"] == "control"):
        configuration["name"] = "control"

    if (configuration["type"] == "dynamos"):
        configuration["name"] = "dynamos"

    node = slice.add_node(name=configuration["name"],
                          site=configuration["site"],
                          host=configuration["host"],
                          cores=configuration["cores"],
                          ram=configuration["ram"],
                          disk=configuration["disk"],
                          validate=True,
                          raise_exception=True,
                          image=image)

    if configuration.get("nvme"):
        # Dedicated NVMe drive; formatted + mounted after the slice is Active.
        node.add_component(model='NVME_P4510', name='nvme1')

    return node

In [ ]:
%%time
# Create a slice
slice = fablib.new_slice(name=slice_name)

# Add Nodes with the specific variables
# Also validate the node can be created and raise an exception in case of failure
print('Adding nodes...')
nodes = [create_node(slice, configuration) for configuration in node_configurations]
nodes_per_site = [
    (site, [node for node in nodes if node.get_site() == site])
    for site in sites
]

print('Adding network interfaces...')
interfaces_per_site = [
    (site, [node.add_component(model='NIC_Basic', name='NIC').get_interfaces()[0] for node in nodes])
    for (site, nodes) in nodes_per_site
]

print('Adding network...')
networks = [
    slice.add_l3network(name=f'Network-{site}', interfaces=interfaces, type="IPv4")
    for (site, interfaces) in interfaces_per_site
]

print(networks, [n.get_gateway() for n in networks], [n.get_subnet() for n in networks])

# Lease: ask for the 2-week maximum explicitly (the default lease is 24h and
# a bare submit dies the next day).
lease_end = datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(days=14)
lease_end_str = lease_end.strftime("%Y-%m-%d %H:%M:%S %z")
print(f"Requesting lease end: {lease_end_str}")

# Polling ~6-7 min (~14 retries) here is normal; if it looks hung, verify the
# nodes with node.execute before recreating the slice.
print('Creating slice...')
# fablib 2.x: submit wants the datetime itself (it strftimes internally);
# the 1.9.x-style formatted string breaks with str-has-no-strftime.
slice.submit(wait=True, wait_timeout=3600, wait_interval=20, progress=True,
             wait_jupyter='text', lease_end_time=lease_end);

**Verify the lease.** The next cell prints the lease end — it must be about
two weeks out. If it shows ~24 hours, the lease request did not take; renew
with `slice.renew(...)` before doing anything else.

In [ ]:
slice = fablib.get_slice(name=slice_name)
print(f"Lease end: {slice.get_lease_end()}")

In [ ]:
%%time
slice = fablib.get_slice(name=slice_name)
nodes = slice.get_nodes()

def get_network_checked(slice, name):
    # fablib 2.x: get_network can return None instead of raising when missing.
    network = slice.get_network(name=name)
    if network is None:
        raise RuntimeError(f"network {name!r} not found in slice")
    return network

nodes_and_network_per_site = [
    (site, [node for node in nodes if node.get_site() == site], get_network_checked(slice, f"Network-{site}"))
    for site in sites
]
networks = [network for (_, _, network) in nodes_and_network_per_site]

nodes_network_ips_per_site = [
    (site, nodes, network, network.get_available_ips(len(nodes)))
    for (site, nodes, network) in nodes_and_network_per_site
]

In [ ]:
%%time
def get_interface_checked(node, network_name):
    # fablib 2.x: get_interface can return None instead of raising when missing.
    interface = node.get_interface(network_name=network_name)
    if interface is None:
        raise RuntimeError(f"{node.get_name()}: no interface on network {network_name!r}")
    return interface

def assign_ip(site, network, available_ips, node):
    interface = get_interface_checked(node, f"Network-{site}")
    address = available_ips.pop(0)
    network_gateway = network.get_gateway()
    network_subnet = network.get_subnet()

    network.allocate_ip(address)
    interface.ip_addr_add(addr=address, subnet=network_subnet)
    node.ip_route_add(subnet=network_subnet, gateway=network_gateway)

    # For the multisite IPv4 connection
    for network in networks:
        node.ip_route_add(subnet=network.get_subnet(), gateway=network_gateway)

    return address

ips = [assign_ip(site, network, ips, node) for (site, nodes, network, ips) in nodes_network_ips_per_site for node in nodes];

In [ ]:
%%time
slice = fablib.get_slice(name=slice_name)
nodes = slice.get_nodes()

for node in nodes:
    ssh_command = node.get_ssh_command().replace(
        "-i /home/fabric/work/fabric_config/slice_key", "-i ~/.ssh/keys/FABRIC-slice_key"
    ).replace(
        "-F /home/fabric/work/fabric_config/ssh_config ", ""
    )

    print(ssh_command);

## Phase 3 — NVMe

Formats and mounts the NVMe drive on every node with `nvme: True` (default:
`server` only). **Run once, after the slice is Active.** The cell skips a
node whose mount already exists, so a re-run is safe — but the format step
itself wipes the drive: first use only.

Manual sequence, not `configure_nvme()`: on fablib 2.0.8 the component's
`get_device_name()` returns None and `configure_nvme()` then targets the
sysfs per-controller alias (`/dev/nvme0c0n1`), which is not a block device
(21.09 sitting). The real namespace device is `/dev/nvme0n1`.

`NVME_MOUNT` must match the mount point in
`node_scripts/install_localpath_provisioner.sh` (its `NVME_MOUNT` default).


In [ ]:
%%time
NVME_MOUNT = "/mnt/nvme"  # must match install_localpath_provisioner.sh
NVME_DEV = "/dev/nvme0n1"

slice = fablib.get_slice(name=slice_name)
for configuration in node_configurations:
    if not configuration.get("nvme"):
        continue
    node = slice.get_node(name=configuration["name"])
    component = node.get_component(name='nvme1')
    if component is None:
        raise RuntimeError(f"{configuration['name']}: NVMe component 'nvme1' not found")
    # fablib 2.0.8: get_device_name() -> None, and configure_nvme() then
    # targets /dev/nvme0c0n1 (sysfs per-controller alias, not a block
    # device). Format + mount manually against the real namespace device.
    stdout, stderr = node.execute(f"lsblk -dn -o NAME,SIZE {NVME_DEV} 2>/dev/null || true")
    if not stdout.strip():
        raise RuntimeError(f"{configuration['name']}: {NVME_DEV} not present on the node")
    stdout, stderr = node.execute(f"mountpoint -q {NVME_MOUNT} && echo MOUNTED || true")
    if "MOUNTED" in stdout:
        print(f"{configuration['name']}: {NVME_MOUNT} already mounted — skipping")
        continue
    # First use only past this point: the format wipes the drive.
    node.execute(
        f"sudo parted -s {NVME_DEV} mklabel gpt mkpart primary ext4 0% 100% && sleep 2 && "
        f"sudo mkfs.ext4 -F {NVME_DEV}p1 && "
        f"sudo mkdir -p {NVME_MOUNT} && sudo mount {NVME_DEV}p1 {NVME_MOUNT} && "
        f"echo '{NVME_DEV}p1 {NVME_MOUNT} ext4 defaults,nofail 0 0' | sudo tee -a /etc/fstab"
    )
    stdout, stderr = node.execute(f"df -h {NVME_MOUNT}")


## Phase 4 — Node prep

Base packages (docker, kubelet/kubeadm/kubectl 1.30 repo, python) on every
node, then a name-keyed `nodes_dict` for everything that follows.

In [ ]:
%%time
print("Uploading the node setup...")
threads = [node.upload_file_thread(local_file_path="node_scripts/node_setup.sh", remote_file_path="setup.sh")
           for node in nodes]
[thread.result() for thread in threads]

In [ ]:
%%time
print("Executing the node setup...")
threads = [node.execute_thread(f"chmod +x setup.sh && ./setup.sh")
           for node in nodes]
[thread.result() for thread in threads]

In [ ]:
%%time
# Name-keyed lookup — NEVER index slice.get_nodes() by position: its order
# is not the node_configurations order.
def get_ip(node):
    interface = get_interface_checked(node, f"Network-{node.get_site()}")
    return interface.get_ip_addr()

nodes_dict = dict()

for node in nodes[:]:
    ip = get_ip(node)
    name = node.get_name()
    nodes_dict[name] = {"ip": ip, "node": node}
    print(f"{name}: {ip}")

print(nodes_dict)

In [ ]:
%%time
nodes_dict["control"]["node"].upload_file(local_file_path="node_scripts/install_k9s.sh", remote_file_path="k9s.sh")
nodes_dict["control"]["node"].execute(f"chmod +x k9s.sh && ./k9s.sh");

## Phase 5 — Kubespray

Inventory is written with the **FABNetv4 IPv4 addresses only** — never the
IPv6 management addresses: kubespray/etcd builds URLs from these values and
IPv6 literals break etcd's URL parsing. The control node gets the slice
private key (for ansible to reach the workers), our `ansible.cfg`, and
`dot_kube.sh` as a utility for restoring kubeconfig after a reset.

In [ ]:
%%time
# FABNet IPv4 addresses ONLY (see the phase note above).
inventory = (
    f"[kube_control_plane]\n"
    f"control ansible_host={nodes_dict['control']['ip']} ip={nodes_dict['control']['ip']} etcd_member_name=etcd1\n"
    f"\n"
    f"[etcd:children]\n"
    f"kube_control_plane\n"
    f"\n"
    f"[kube_node]\n"
    f"dynamos ansible_host={nodes_dict['dynamos']['ip']} ip={nodes_dict['dynamos']['ip']}\n"
)

for node_name in nodes_dict.keys():
    if node_name not in ["dynamos", "control"]:
        inventory += f"{node_name} ansible_host={nodes_dict[node_name]['ip']} ip={nodes_dict[node_name]['ip']}\n"

with open('kubespray/inventory.ini', 'w') as f:
    f.write(inventory)

In [ ]:
print(inventory)

In [ ]:
%%time
nodes_dict['control']['node'].upload_file(local_file_path="node_scripts/control_kubespray_setup.sh", remote_file_path="kubespray_setup.sh");
nodes_dict['control']['node'].execute("chmod +x kubespray_setup.sh && ./kubespray_setup.sh");

nodes_dict['control']['node'].upload_file(local_file_path="kubespray/inventory.ini", remote_file_path="kubespray/inventory/dynamos/inventory.ini");
nodes_dict['control']['node'].upload_file(local_file_path="kubespray/ansible.cfg", remote_file_path="kubespray/ansible.cfg");
nodes_dict['control']['node'].upload_file(local_file_path="node_scripts/dot_kube.sh", remote_file_path="dot_kube.sh");
nodes_dict['control']['node'].execute("chmod +x ./dot_kube.sh");
nodes_dict['control']['node'].upload_file(local_file_path="/home/fabric/work/fabric_config/slice_key", remote_file_path="/home/ubuntu/.ssh/slice_key");
nodes_dict['control']['node'].execute("chmod 600 /home/ubuntu/.ssh/slice_key");

In [ ]:
%%time
# 27 mins, 20 mins, 10 min at CIEN
nodes_dict['control']['node'].upload_file(local_file_path="node_scripts/start_kubespray.sh", remote_file_path="start.sh");
nodes_dict['control']['node'].execute(f"chmod +x start.sh && ./start.sh");

In [ ]:
# This is for resetting the kubespray cluster.
# Use this if you are troubleshooting your Kubernetes cluster
# and you want to redeploy fresh.

# nodes_dict['control']['node'].upload_file(local_file_path="node_scripts/reset_kubespray.sh", remote_file_path="reset.sh");
# nodes_dict['control']['node'].execute(f"chmod +x reset.sh && ./reset.sh");

## Phase 6 — Storage provisioner

Kubespray installs **no** storage provisioner, and the DYNAMOS charts' etcd
and rabbit PVCs name no `storageClassName` — they ride the cluster default.
The namespaces chart ships the `gateway-state` StorageClass with provisioner
`rancher.io/local-path`. `install_localpath_provisioner.sh` therefore:

- installs rancher local-path-provisioner **v0.0.37**,
- backs `gateway-state` claims on the `server` node with the NVMe mount,
- marks `local-path` as the cluster default StorageClass.

Run after the cluster is up, before the DYNAMOS install.

In [ ]:
%%time
nodes_dict['control']['node'].upload_file(local_file_path="node_scripts/install_localpath_provisioner.sh", remote_file_path="install_localpath_provisioner.sh");
nodes_dict['control']['node'].execute(f"chmod +x install_localpath_provisioner.sh && NVME_MOUNT={NVME_MOUNT} ./install_localpath_provisioner.sh");

## Phase 7 — DYNAMOS

**Repo delivery is switchable.** `DYNAMOS_REPO_URL = ""` (the default) means
the **upload route**: the next cell pushes the local checkout's `charts/`,
`configuration/` and the `scripts/add_*.sh` helpers to `control:~/DYNAMOS`.
The fork may be private, and the upload route needs no token — it does need
this notebook to run from `fabric/` inside a full checkout of the fork.
Setting `DYNAMOS_REPO_URL` to a clone URL makes `install_dynamos.sh` clone
it instead.

The etcd seed always comes from the local checkout's
`configuration/etcd_launch_files` (it carries the `state_aware_*.json` files)
— never from the old git-clone seed variant.

In [ ]:
%%time
# "" -> UPLOAD route (default). Non-empty -> install_dynamos.sh clones it.
DYNAMOS_REPO_URL = ""

control = nodes_dict['control']['node']

if not DYNAMOS_REPO_URL:
    print("Upload route: pushing charts/, configuration/ and scripts/ (incl. the E3 harness + digest gate) to control:~/DYNAMOS ...")
    # upload_directory recreates the local directory name inside the remote path
    control.upload_directory(local_directory_path="../charts", remote_directory_path="/home/ubuntu/DYNAMOS")
    control.upload_directory(local_directory_path="../configuration", remote_directory_path="/home/ubuntu/DYNAMOS")
    # The whole scripts/ dir: add_* for the charts, plus the ported E3
    # harness and scripts/e3_fabric_digests.txt (the counted-run gate) —
    # the smoke/probe/counted sessions run from control:~/DYNAMOS.
    control.upload_directory(local_directory_path="../scripts", remote_directory_path="/home/ubuntu/DYNAMOS")
    control.execute("chmod +x /home/ubuntu/DYNAMOS/scripts/*.sh /home/ubuntu/DYNAMOS/configuration/*.sh")
    # Root-level scripts the harness reaches for (capture_lib.sh calls
    # ~/DYNAMOS/reset.sh). The archive includes them; upload to root.
    control.upload_file(local_file_path="../reset.sh", remote_file_path="/home/ubuntu/DYNAMOS/reset.sh")
    control.upload_file(local_file_path="../fix-rabbit.sh", remote_file_path="/home/ubuntu/DYNAMOS/fix-rabbit.sh")
    control.execute("ls /home/ubuntu/DYNAMOS")
else:
    print(f"Clone route: install_dynamos.sh will clone {DYNAMOS_REPO_URL}")

In [ ]:
%%time
# etcd seed — ALWAYS the local-upload variant. (define_etcd_data.sh, the
# git-clone seed, pulls a stale repo and must not be used.)
dynamos_node = nodes_dict['dynamos']['node']
dynamos_node.upload_directory(local_directory_path="../configuration/etcd_launch_files", remote_directory_path="./")
dynamos_node.execute("ls etcd_launch_files")

dynamos_node.upload_file(local_file_path="node_scripts/define_etcd_data_local.sh", remote_file_path="define_etcd_data_local.sh");
dynamos_node.execute(f"chmod +x define_etcd_data_local.sh && ./define_etcd_data_local.sh");

In [ ]:
%%time
# Helm + Linkerd (edge-25.8.1) + gateway-api v1.2.1 + Jaeger + etcd client on
# control, plus the repo via the chosen route. If `linkerd check --pre` fails
# on known already-exists leftovers only, re-run with LINKERD_PRECHECK=skip
# (see install_dynamos.sh).
control.upload_file(local_file_path="node_scripts/install_dynamos.sh", remote_file_path="dynamos.sh");
control.execute(f"chmod +x dynamos.sh && DYNAMOS_REPO_URL='{DYNAMOS_REPO_URL}' ./dynamos.sh")

**`configure_dynamos.sh` — kept for parity, disabled for this fork.** It must
run *after* the repo-delivery cell (SD's saved outputs show it failing when
run before the clone existed). But in this fork, `charts/agents/templates/`
already ships `server/clientone/clienttwo/clientthree.yaml` with the fork's
own image settings (`MICROSERVICE_REPOSITORY_NAME`/`SIDECAR_REPOSITORY_NAME`,
templated with a `tamjiyan` default and re-pointed to `2bias123` by the
install cell's `DYNAMOS_IMG_ACCOUNT`, plus `LOCAL_DEV`), and all four namespace + cluster-role entries are checked in.
`add_agent.sh` would **regenerate those four templates from `workerX.yaml`**
and silently lose the fork's values. Enable only to add an agent the charts
do not already carry.

In [ ]:
# Ordering: must come AFTER the repo-delivery cell (fails when ~/DYNAMOS
# does not exist yet). Disabled for this fork — see the note above.
agents_string = ",".join(agents)
thirdparties_string = ",".join(thirdparties)
print(f'configure_dynamos.sh {agents_string} "{thirdparties_string}" — disabled for this fork, see note above')

# nodes_dict['control']['node'].upload_file(local_file_path="node_scripts/configure_dynamos.sh", remote_file_path="configure_dynamos.sh");
# nodes_dict['control']['node'].execute(f"chmod +x configure_dynamos.sh && ./configure_dynamos.sh {agents_string} {thirdparties_string}");

In [ ]:
%%time
# Optionally refresh the two installer scripts from the local checkout first
# (useful on the clone route, or after local edits). If you do, re-run the
# sed below afterwards.
# control.upload_file(local_file_path="../configuration/dynamos-configuration.sh", remote_file_path="/home/ubuntu/DYNAMOS/configuration/dynamos-configuration.sh")
# control.upload_file(local_file_path="../configuration/fill-rabbit-pvc.sh", remote_file_path="/home/ubuntu/DYNAMOS/configuration/fill-rabbit-pvc.sh")

# Fork fix-up, applied to the control node's copy only (the local repo is
# never edited): configuration/dynamos-configuration.sh hardcodes
# DYNAMOS_ROOT="${HOME}/Thesis/DYNAMOS", but on FABRIC the repo lives at
# ~/DYNAMOS.
control.execute("sed -i 's|/Thesis/DYNAMOS|/DYNAMOS|' /home/ubuntu/DYNAMOS/configuration/dynamos-configuration.sh")
control.execute("grep -n 'DYNAMOS_ROOT=' /home/ubuntu/DYNAMOS/configuration/dynamos-configuration.sh");

In [ ]:
%%time
# Install DYNAMOS. SUDO=sudo because the fork's dynamos-configuration.sh
# hashes the RabbitMQ password via `$SUDO docker run ...` and the ubuntu
# user is not in the docker group.
# DYNAMOS_IMG_ACCOUNT/TAG re-point every fork image to the 2bias123 Hub
# account (e3-balanced build; digests pinned in scripts/e3_fabric_digests.txt).
# The job-deploy path additionally pulls :latest aliases of the same digests.
control.execute("SUDO=sudo DYNAMOS_IMG_ACCOUNT=2bias123 DYNAMOS_IMG_TAG=e3-balanced ~/DYNAMOS/configuration/dynamos-configuration.sh")

## Operate — run E3 on the deployed cluster

Phases 1–7 deploy the cluster **once**. Everything below operates the
already-running slice: reconnect after a kernel restart, refresh the harness
on control, then run the smoke / probe / counted E3 sessions.

**Do not re-run Phases 1–7 on a live cluster** — they recreate the slice and
redeploy DYNAMOS. Only the cells here (and the Phase 7 repo-delivery cell,
when you need to push updated scripts) are safe to run against a live slice.

In [ ]:
# Reconnect to the running slice (run after any kernel restart).
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
fablib = fablib_manager()
slice_name = 'DYNAMOS-E3-MAX'
slice = fablib.get_slice(name=slice_name)
print("lease end:", slice.get_lease_end())

def _iface(node, net):
    itf = node.get_interface(network_name=net)
    if itf is None:
        raise RuntimeError(f"{node.get_name()}: no interface on {net!r}")
    return itf

nodes_dict = {}
for node in slice.get_nodes():
    ip = _iface(node, f"Network-{node.get_site()}").get_ip_addr()
    nodes_dict[node.get_name()] = {"ip": ip, "node": node}
control = nodes_dict['control']['node']
print("nodes:", {n: nodes_dict[n]['ip'] for n in nodes_dict})

In [ ]:
# Refresh the harness on control: push this checkout's scripts/ + reset.sh.
# Safe to re-run (file uploads only, no redeploy). Do this after any kit update.
control.upload_directory(local_directory_path="../scripts", remote_directory_path="/home/ubuntu/DYNAMOS")
control.upload_file(local_file_path="../reset.sh", remote_file_path="/home/ubuntu/DYNAMOS/reset.sh")
control.upload_file(local_file_path="../fix-rabbit.sh", remote_file_path="/home/ubuntu/DYNAMOS/fix-rabbit.sh")
control.execute("chmod +x /home/ubuntu/DYNAMOS/scripts/*.sh /home/ubuntu/DYNAMOS/reset.sh /home/ubuntu/DYNAMOS/fix-rabbit.sh")
control.execute("ls /home/ubuntu/DYNAMOS/scripts | head")

In [ ]:
# Pre-warm the chain images on every worker node so no run races a cold pull
# (the counted run pins from a warm cache; the smoke also runs faster).
JOB_IMAGES = [
    "2bias123/agent:latest", "2bias123/sidecar:latest",
    "2bias123/policy-checkpoint-gateway:latest",
    "2bias123/vfl-train-demo:latest", "2bias123/vfl-train-model-demo:latest",
    "2bias123/anonymize-test:latest", "2bias123/differential-p-test:latest",
]
for name in ["dynamos", "server", "clientone", "clienttwo", "clientthree"]:
    cmd = " ; ".join(f"sudo crictl pull {img}" for img in JOB_IMAGES)
    nodes_dict[name]["node"].execute(cmd, quiet=True)
    print("prewarmed", name)

In [ ]:
# SMOKE (uncounted): one Williams quadruplet (a,b,h,c) at 5 rounds.
# Runs run_e3_balanced.sh directly, bypassing the digest-equality wrapper
# (its job-image gate is unfinished); the env-snapshot provenance gate still
# runs. POLL_MAX=180 absorbs a cold first-job pull.
out, err = control.execute(
    "cd ~/DYNAMOS && QUADS=1 CYCLES=5 POLL_MAX=180 "
    "E3_ENV=fabric E3_TRANSPORT=nodeport E3_CROSSLOG_ASSERT=record "
    "E3_RESULTS_DIR=/tmp/e3-smoke ./scripts/run_e3_balanced.sh", quiet=True)
print(out[-6000:]); print("ERR", err[-1500:])

In [ ]:
# TIMING PROBE (uncounted): one quadruplet at 50 rounds. Prices the session
# so you can set POLL_MAX and confirm the counted n. Read the per-round
# timings ('steady state ...') from the output.
out, err = control.execute(
    "cd ~/DYNAMOS && QUADS=1 CYCLES=50 POLL_MAX=300 "
    "E3_ENV=fabric E3_TRANSPORT=nodeport E3_CROSSLOG_ASSERT=record "
    "E3_RESULTS_DIR=/tmp/e3-probe ./scripts/run_e3_balanced.sh", quiet=True)
print(out[-8000:]); print("ERR", err[-1500:])

In [ ]:
# COUNTED SESSION: 32 Williams quadruplets at 50 rounds via the frozen-build
# wrapper (digest gate). Launch inside tmux on control so it survives a
# disconnect, then poll the log. Set POLL_MAX from the probe.
#   PREREQ: the wrapper's digest gate needs the job-image fix (see runbook);
#   until then run via run_e3_balanced.sh direct as the smoke/probe cells do.
LAUNCH = (
    "cd ~/DYNAMOS && tmux new -s e3 -d "
    "'QUADS=32 CYCLES=50 POLL_MAX=300 E3_ENV=fabric E3_TRANSPORT=nodeport "
    "E3_CROSSLOG_ASSERT=record E3_RESULTS_DIR=/tmp/e3-fabric "
    "./scripts/run_e3_fabric_32quads.sh > /tmp/e3-counted.log 2>&1'"
)
print("Launch (edit POLL_MAX first, run when the digest-gate fix is in):")
print(LAUNCH)
# out, err = control.execute(LAUNCH); print(out, err)          # uncomment to launch
# out, err = control.execute("tail -40 /tmp/e3-counted.log")   # poll progress

In [ ]:
# Pull a results tree off the slice to JupyterHub, then download via the file
# browser. Change the name for smoke / probe / counted (e3-smoke|e3-probe|e3-fabric).
TREE = "e3-smoke"
control.execute(f"cd /tmp && tar czf {TREE}.tgz {TREE} && ls -la /tmp/{TREE}.tgz")
control.download_file(f"/home/fabric/work/{TREE}.tgz", f"/tmp/{TREE}.tgz")
print(f"pulled to ~/work/{TREE}.tgz — download it via the file browser")

## Phase 8 — Teardown

`helm uninstall` does **not** delete the namespaces the charts created —
follow up with `kubectl delete namespace` for a truly clean slate. The
release names below match this fork's `dynamos-configuration.sh` (`surf` is
the thirdparty release; nginx lives in the `ingress` namespace).

In [ ]:
# Optional to clean up: uninstall DYNAMOS

command = "helm uninstall agents api-gateway core orchestrator namespaces prometheus surf"
# nodes_dict['control']['node'].execute(command)
# nodes_dict['control']['node'].execute("helm uninstall -n ingress nginx")

# helm keeps the namespaces — delete them too for a clean slate:
# nodes_dict['control']['node'].execute("kubectl delete namespace core orchestrator api-gateway ingress server clientone clienttwo clientthree uva vu surf")

In [ ]:
# Optional delete etcd/rabbit PVCs
# nodes_dict['control']['node'].execute("kubectl get pvc --all-namespaces")

# nodes_dict['control']['node'].execute("kubectl delete pvc etcd-data-etcd-0 -n core")
# nodes_dict['control']['node'].execute("kubectl delete pvc etcd-data-etcd-1 -n core")
# nodes_dict['control']['node'].execute("kubectl delete pvc etcd-data-etcd-2 -n core")
# nodes_dict['control']['node'].execute("kubectl delete pvc rabbit-pvc rabbitmq-data-pvc rabbitmq-log-pvc -n core")
# nodes_dict['control']['node'].execute("kubectl delete pvc etcd-pvc -n orchestrator")

# nodes_dict['control']['node'].execute("kubectl get pvc --all-namespaces")

In [ ]:
# Final teardown: delete the slice itself (frees all FABRIC resources,
# including the NVMe drive and its contents).
# slice = fablib.get_slice(name=slice_name)
# slice.delete()